# Лабораторная работа 1. Вариант 1
**Тема.** Базовые задачи оптимизации.
</br>
### **Задача 1.** 
Решение задачи минимизации взвешенной суммы моментов завершения работ на одном станке.

$$
1 \space | \space | \sum w_i C_i 
$$

где
- 1: один исполнитель (одна машина).
- Пустое поле: нет никаких ограничений. Задачи можно прерывать, они все доступны сразу ($r_i = 0$), у них нет зависимостей друг от друга.
- $\sum w_i C_i $: целевая функция.
    - $C_i$: это время завершения конкретной задачи $i$.
    - $w_i$: это вес или штраф за каждую единицу времени, пока задача не готова.

**Задание:** минимизировать суммарную стоимость ожидания всех задач.

Для решения таких задач применяется **WSPT** или [**правило Смита**](https://web-static.stern.nyu.edu/om/faculty/pinedo/scheduling/shakhlevich/handout02.pdf) [1, 2.4]:

> Работы должны быть упорядочены по возрастанию отношения веса к длительности $ \frac{p_j}{w_j}$

Определим условия.

In [1]:
jobs = [
    (12, 8, "Job 1"),
    (7, 3, "Job 2"),
    (10, 12, "Job 3"),
    (4, 1, "Job 4"),
    (21, 7, "Job 5"),
    (17, 5, "Job 6"),
    (13, 3, "Job 7")
]

Сортировка по правилу **WSPT**: отношение $\frac{w_i}{p_i}$ по возрастанию

In [2]:
sorted_jobs = sorted(jobs, key=lambda x: x[0]/x[1])

Смоделируем график работ

In [3]:
current_time = 0
total_weighted_completion_time = 0
schedule_details = []

for w, p, name in sorted_jobs:
    current_time += p 
    total_weighted_completion_time += w * current_time
    schedule_details.append({
        'Job': name,
        'w': w,
        'p': p,
        'C': current_time,
        'w*C': w * current_time,
        'w/p': w/p 
    })

Выведем результаты

In [4]:
print(f"Минимальное значение sum(w_i * C_i): {total_weighted_completion_time}")
print("-" * 30)
for entry in schedule_details:
    print(f"{entry['Job']}: C = {entry['C']}, w*C = {entry['w*C']},  w/p = {entry['w/p']:.3}")

Минимальное значение sum(w_i * C_i): 2397
------------------------------
Job 3: C = 12, w*C = 120,  w/p = 0.833
Job 1: C = 20, w*C = 240,  w/p = 1.5
Job 2: C = 23, w*C = 161,  w/p = 2.33
Job 5: C = 30, w*C = 630,  w/p = 3.0
Job 6: C = 35, w*C = 595,  w/p = 3.4
Job 4: C = 36, w*C = 144,  w/p = 4.0
Job 7: C = 39, w*C = 507,  w/p = 4.33


### **Задача 2.** 
Решить задачу
$$
1 \space | prec | \sum w_i C_i
$$
при следующем графе предшествования
$$
1 \rightarrow 4 \\
2 \rightarrow 6 \rightarrow 3 \rightarrow 8 \\
5 \rightarrow 7
$$
Для решения такого рода задач применяется **метод цепочек** [1, 2.5]:
- из каждого набора работ в графе сформировать набор нераспределнных работ;
- найти минимальный $\rho$-фактор для каждой цепи, где

$$
\rho = \frac{\sum p_i}{\sum w_i}
$$

- выбрать работы с наименьшим $\rho$;
- поместить работы в план, удалив их из графа

Метод является циклическим, то есть шаги посторяются до тех пор, пока граф не опустеет.

Определим граф предшествования и цепи внутри него

In [5]:
w = {1: 12, 2: 7, 3: 10, 4: 4, 5: 21, 6: 17, 7: 13}
p = {1: 8, 2: 3, 3: 12, 4: 1, 5: 7, 6: 5, 7: 3}
chains = [(1, 4), (2, 6, 3), (5, 7)]

In [6]:
remaining_chains = [list(c) for c in chains]
final_schedule = []

Выполняем цикл метода, пока в графе остаются нераспределенные работы.

In [7]:
while any(remaining_chains):
    best_rho = -1
    best_segment = []
    best_chain_idx = -1
    
    for idx, chain in enumerate(remaining_chains):
        if not chain: continue
            
        current_sum_w = 0
        current_sum_p = 0
        for i in range(len(chain)):
            job = chain[i]
            current_sum_w += w[job]
            current_sum_p += p[job]
            rho = current_sum_w / current_sum_p
            
            if rho > best_rho:
                best_rho = rho
                best_segment = chain[:i+1]
                best_chain_idx = idx
    final_schedule.extend(best_segment)
    remaining_chains[best_chain_idx] = remaining_chains[best_chain_idx][len(best_segment):]

Выведем получившийся план

In [8]:
print(f"Оптимальный порядок работ: {final_schedule}")

Оптимальный порядок работ: [5, 7, 2, 6, 1, 4, 3]


## References
1. ALGORITHMS FOR SEQUENCING AND SCHEDULING, Ibrahim M. Alharkan